In [15]:
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

In [9]:
df = pd.read_csv("data/spam.csv",encoding='latin-1')[['v1','v2']]
df.columns=['label','message']
df['label']=df['label'].map({'ham':0,'spam':1})

In [3]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nikhi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

In [10]:
df['cleaned'] = df['message'].apply(preprocess)

print("Before:", df['message'][2])
print("After: ", df['cleaned'][2])


Before: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
After:  free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply overs


In [14]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [16]:


# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_f1 = f1_score(y_test, lr.predict(X_test))

# LinearSVC
svc = LinearSVC(max_iter=1000)
svc.fit(X_train, y_train)
svc_f1 = f1_score(y_test, svc.predict(X_test))

print(f"Logistic Regression F1: {lr_f1:.3f}")
print(f"LinearSVC F1:           {svc_f1:.3f}")

Logistic Regression F1: 0.773
LinearSVC F1:           0.904


In [17]:
print("\n--- Model Comparison ---")
print(f"{'Model':<25} {'Vectorizer':<15} {'F1'}")
print(f"{'Logistic Regression':<25} {'CountVectorizer':<15} 0.910")  # from Day 8
print(f"{'Logistic Regression':<25} {'TF-IDF':<15} {lr_f1:.3f}")
print(f"{'LinearSVC':<25} {'TF-IDF':<15} {svc_f1:.3f}")


--- Model Comparison ---
Model                     Vectorizer      F1
Logistic Regression       CountVectorizer 0.910
Logistic Regression       TF-IDF          0.773
LinearSVC                 TF-IDF          0.904


In [19]:
import numpy as np

feature_names = tfidf.get_feature_names_out()
coefficients = lr.coef_[0]

# Top spam words (highest positive coefficients)
top_spam_idx = np.argsort(coefficients)[-5:][::-1]
print("Top 20 SPAM words:")
for idx in top_spam_idx:
    print(f"  {feature_names[idx]:<15} {coefficients[idx]:.3f}")

# Top ham words (highest negative coefficients)
top_ham_idx = np.argsort(coefficients)[:5]
print("\nTop 20 HAM words:")
for idx in top_ham_idx:
    print(f"  {feature_names[idx]:<15} {coefficients[idx]:.3f}")

Top 20 SPAM words:
  txt             4.784
  stop            3.856
  free            3.845
  claim           3.837
  mobile          3.664

Top 20 HAM words:
  ltgt            -1.973
  ok              -1.882
  im              -1.878
  ill             -1.856
  sir             -1.473
